In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd

from helpers.features import get_processed_dfs
from process_dumb_models import get_model_results

In [ ]:
train_df = pd.read_parquet("../Stamford/Data/train.parquet")
train_df, train_df_label = train_df.drop(columns = ["label"]), train_df["label"]
val_df = pd.read_parquet("../Stamford/Data/val.parquet")
val_df, val_df_label = val_df.drop(columns = ["label"]), val_df["label"]
test_df = pd.read_parquet("../Stamford/Data/test.parquet")
test_df, test_df_label = test_df.drop(columns = ["label"]), test_df["label"]

# Same representation as Overture: StreetName = "directional name suffix" (e.g. "S HACKETT CIR").
# number_missing is computed first, from the original house-number columns.
for df in [train_df, val_df, test_df]:
    df["both_number_missing"] = df["Address_a"].isna() & df["Address_b"].isna()
    df["number_missing"] = (df["Address_a"].isna() | df["Address_b"].isna()) & ~df["both_number_missing"]
    df["directional_a"] = df["directional_a"].fillna("")
    df["directional_b"] = df["directional_b"].fillna("")

    df["StreetName_a"] = (df["directional_a"] + " "+df["StreetName_a"] + " " +df["street_extension_a"]).str.strip()
    df["StreetName_b"] = (df["directional_b"] + " "+df["StreetName_b"] + " " + df["street_extension_b"]).str.strip()

In [ ]:
# Stamford features (cached) + model run
train_stamford, val_stamford, test_stamford = get_processed_dfs([
    [train_df, "./stamford_training/evaluation_train.parquet"],
    [val_df, "./stamford_training/evaluation_val.parquet"],
    [test_df, "./stamford_training/evaluation_test.parquet"]
])

# Stamford numbers must be re-scored: data regenerated 2026-10-11.
# get_model_results(train_stamford, train_df_label, val_stamford, val_df, val_df_label)